<a href="https://colab.research.google.com/github/ankitraj74304-hue/Ankit_raj/blob/main/Dijkstra_lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import heapq
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter
import networkx as nx #function Animation ka yeh kam hain  frames ko animate karta hain  or pillowritwer ka yeh kam hain  jo animation ko GIF me save karta hain

INF = float("inf") #infinity  (source ke alawa kisi node tak  distance pata nhi hain)

EDGES = [ #graph me vertices/ node ke  bich k  ka connection link hota hain]
    ("A", "B", 4), ("A", "C", 1), ("C", "B", 2), ("B", "D", 5),
    ("C", "D", 8), ("C", "E", 10), ("D", "E", 2), ("D", "F", 6),
    ("E", "F", 3), ("E", "G", 7), ("F", "G", 1), ("F", "H", 4),
    ("G", "H", 2)
]

SOURCE = "A" # shortest path calcuation hain jo A se start hoti hain]

NODES = sorted({u for u, v, w in EDGES} | {v for u, v, w in EDGES}) # saare starting node U or ending  nodes V hain

DIRECTED_EDGES = []
for u, v, w in EDGES:
    DIRECTED_EDGES.append((u, v, w))
    DIRECTED_EDGES.append((v, u, w))

ADJ = {n: [] for n in NODES} # ADJ yeh ek adjacency list banata hain
for u, v, w in DIRECTED_EDGES:
    ADJ[u].append((v, w))

def dijkstra_steps(source): # yeh Dijkstra  algorithm ko execute karta hain
#lekin yeh direct result retrun nhi karta hain
    dist = {n: INF for n in NODES}
    pred = {n: None for n in NODES} # yeh kis node tak shortest path me uske just previous  node ko store karega
    dist[source] = 0 # source ka distance kitna hain
    visited = set() # visited nodes ka record  rakhta hain
    pq = [(0, source)] # priority queue  vo hain jisme element priority ke basics par out (niklta) hain

    yield dict(dist=dist.copy(), pred=pred.copy(), visited=visited.copy(), current=None,
               relaxed=None, note="Start")

    while pq:
        d, u = heapq.heappop(pq)
        if u in visited:
            continue
        visited.add(u)
        yield dict(dist=dist.copy(), pred=pred.copy(), visited=visited.copy(), current=u,
                   relaxed=None, note=f"Visit {u} (finalized dist={d})")

        for v, w in ADJ[u]:
            if v in visited:
                continue
            newd = d + w
            if newd < dist[v]:
                dist[v] = newd
                pred[v] = u # yeh pr update hua hain
                heapq.heappush(pq, (newd, v))
                yield dict(dist=dist.copy(), pred=pred.copy(), visited=visited.copy(), current=u,
                           relaxed=(u, v), note=f"Relax {u}->{v}: dist[{v}] = {newd}")

    yield dict(dist=dist.copy(), pred=pred.copy(), visited=visited.copy(), current=None,
               relaxed=None, note="Dijkstra done")

def bellman_ford_steps(source):
    dist = {n: INF for n in NODES}
    pred = {n: None for n in NODES} # yeh par  Initialize hua hain
    dist[source] = 0
    yield dict(dist=dist.copy(), pred=pred.copy(), current=None, relaxed=None,
               note="Start", iteration=0)

    n_nodes = len(NODES)
    for i in range(1, n_nodes):
        updated = False
        for (u, v, w) in DIRECTED_EDGES:
            if dist[u] == INF:
                continue
            newd = dist[u] + w
            if newd < dist[v]:
                dist[v] = newd
                pred[v] = u  #yeha par  update hua hain
                updated = True
                yield dict(dist=dist.copy(), pred=pred.copy(), current=u, relaxed=(u, v),
                           note=f"Pass {i}: Relax {u}->{v} = {newd}", iteration=i)
        if not updated:
            yield dict(dist=dist.copy(), pred=pred.copy(), current=None, relaxed=None,
                       note=f"Pass {i}: no changes -> converged early", iteration=i)
            break

    neg_cycle = any(
        dist[u] != INF and dist[u] + w < dist[v]
        for (u, v, w) in DIRECTED_EDGES
    )
    yield dict(dist=dist.copy(), pred=pred.copy(), current=None, relaxed=None,
               note="Negative cycle detected!" if neg_cycle else "Bellman-Ford done",
               iteration=n_nodes)

dij_frames = list(dijkstra_steps(SOURCE))
bf_frames = list(bellman_ford_steps(SOURCE))
total_frames = max(len(dij_frames), len(bf_frames))

G = nx.Graph()
G.add_weighted_edges_from(EDGES)
pos = nx.spring_layout(G, seed=42)
edge_labels = nx.get_edge_attributes(G, "weight")


def draw_panel(ax, frame, title, visited_set=None):
    ax.clear()
    node_colors = []
    for n in G.nodes():
        if n == SOURCE:
            node_colors.append("gold")
        elif visited_set is not None and n in visited_set:
            node_colors.append("lightgreen")
        elif n == frame["current"]:
            node_colors.append("orange")
        else:
            node_colors.append("lightblue")

    edge_colors, edge_widths = [], []
    for (u, v) in G.edges():
        if frame["relaxed"] and {u, v} == set(frame["relaxed"]):
            edge_colors.append("red")
            edge_widths.append(3.0)
        else:
            edge_colors.append("gray")
            edge_widths.append(1.0)

    nx.draw_networkx_nodes(G, pos, ax=ax, node_color=node_colors, node_size=650)
    nx.draw_networkx_edges(G, pos, ax=ax, edge_color=edge_colors, width=edge_widths)
    nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, ax=ax, font_size=8)

    for n, (x, y) in pos.items():
        d = frame["dist"][n]
        label = f"{n}\n{d if d != INF else chr(0x221E)}"
        ax.text(x, y, label, ha="center", va="center", fontsize=9, fontweight="bold")

    ax.set_title(title, fontsize=10)
    ax.axis("off")


fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 6.5))


def update(i):
    f1 = dij_frames[min(i, len(dij_frames) - 1)]
    f2 = bf_frames[min(i, len(bf_frames) - 1)]
    draw_panel(ax1, f1, f"Dijkstra — {f1['note']}", visited_set=f1["visited"])
    draw_panel(ax2, f2, f"Bellman-Ford — {f2['note']}", visited_set=None)
    fig.suptitle(f"Single Source Shortest Path from '{SOURCE}'  |  frame {i + 1}/{total_frames}")


anim = FuncAnimation(fig, update, frames=total_frames, interval=1000, repeat=False)

output_path = "dijkstra_bellman_animation.gif"
anim.save(output_path, writer=PillowWriter(fps=1))
print(f"Saved animation to {output_path}")

final_dij = dij_frames[-1]["dist"]
final_bf = bf_frames[-1]["dist"]

final_dij_pred = dij_frames[-1]["pred"]
final_bf_pred = bf_frames[-1]["pred"]

print(f"\nShortest distances from source '{SOURCE}':")
print(f"{'Node':<6}{'Dijkstra':<12}{'Bellman-Ford':<14}{'Match?'}")
for n in NODES:
    d1 = final_dij[n]
    d2 = final_bf[n]
    print(f"{n:<6}{d1:<12}{d2:<14}{'OK' if d1 == d2 else 'MISMATCH'}")

print(f"\nPredecessor nodes for Dijkstra's from source '{SOURCE}':")
for n in NODES:
    print(f"To {n}: {final_dij_pred[n] if final_dij_pred[n] is not None else 'None'}")

print(f"\nPredecessor nodes for Bellman-Ford from source '{SOURCE}':")
for n in NODES:
    print(f"To {n}: {final_bf_pred[n] if final_bf_pred[n] is not None else 'None'}")

Saved animation to dijkstra_bellman_animation.gif

Shortest distances from source 'A':
Node  Dijkstra    Bellman-Ford  Match?
A     0           0             OK
B     3           3             OK
C     1           1             OK
D     8           8             OK
E     10          10            OK
F     13          13            OK
G     14          14            OK
H     16          16            OK

Predecessor nodes for Dijkstra's from source 'A':
To A: None
To B: C
To C: A
To D: B
To E: D
To F: E
To G: F
To H: G

Predecessor nodes for Bellman-Ford from source 'A':
To A: None
To B: C
To C: A
To D: B
To E: D
To F: E
To G: F
To H: G


In [ ]:

INF = float('inf') #infinity
edges = [ # yeh par hum graph ki edges define karte  hain
    ('A', 'B', 2), # jaise ki Ase B jane me cost/weight 4 lag rahi hain
    ('A', 'C', 1),# A to C cost 2 lag rahi ahin
    ('C', 'B', 5),# C se B  jane me cost 1 lag rahi hain
    ('B', 'D', 2),# B se D jane me 3 cost  lag raha hain
    ('C', 'D', 6)# C se D jane me 5 cost lag rahin hain
]

vertices = ['A', 'B', 'C', 'D'] # saare grahp ka nodes/ vertices hain
source = 'A' # A se start ho rahi hain
dist = {v: INF for v in vertices} # har vertex ki starting distance infinity set hain
dist[source] = 0
for i in range(len(vertices) - 2):

    for u, v, w in edges: # graph ki har ek ek kr ke check karnge

        if dist[u] != INF and dist[u] + w < dist[v]: # U ke through V tak ka jane ka
            dist[v] = dist[u] + w
negative_cycle = False # es me koi negative cycle nhi hain esliye False hain

for u, v, w in edges: # fir se node check ho rahi hain
    if dist[u] != INF and dist[u] + w < dist[v]: # es me koi bhi distance small ho sakti hain
        negative_cycle = True # graph me negative cycle ho sakti hain
        break # negative cycle milte hi loop ko stop kar deng
print("Source:", source)
print("\nShortest distances:")

for v in vertices: # har vertex ko ek ek karke lenge
    print(source, "->", v, "=", dist[v]) # A se suru current node v tak  print krnge

if negative_cycle: # check karnge gye ki negative  weight cycle mila hain ki nhi
    print("\nNegative weight cycle exists!")
else:
    print("\nNo negative weight cycle") #es me no negative cycle hai  koi negative cycle nahi hain

Source: A

Shortest distances:
A -> A = 0
A -> B = 2
A -> C = 1
A -> D = 4

No negative weight cycle


In [ ]:
class Edge: # es me ek class bani hui hain es class ka use graph ke ek edge connection ko represent  karne ke liye hoga
    def __init__(self, src, dest, weight): # ek constructor jab bhi edge  ka object banyega yeh automatically call hoga
        self.src = src # source vertex
        self.dest = dest # destination vertex
        self.weight = weight # edge ki distance/ cost

    def __repr__(self):
        return f"Edge(src={self.src}, dest={self.dest}, weight={self.weight})"
edge1 = Edge('A', 'B', 4)
print(edge1)

Edge(src=A, dest=B, weight=4)


In [ ]:
INF = float('inf')

class Graph:
    def __init__(self, vertices):
        self.V = vertices
        self.graph = [] # Stores edges in (u, v, w) format
        self.nodes = {node for node in self.V}

    def add_edge(self, u, v, w):
        self.graph.append([u, v, w])
        self.nodes.add(u)
        self.nodes.add(v)

    def bellman_ford(self, source):
        dist = {node: INF for node in self.nodes}
        pred = {node: None for node in self.nodes}
        dist[source] = 0

        # Relax all edges |V| - 1 times
        for _ in range(len(self.nodes) - 1):
            for u, v, w in self.graph:
                if dist[u] != INF and dist[u] + w < dist[v]:
                    dist[v] = dist[u] + w
                    pred[v] = u

        # Check for negative-weight cycles
        negative_cycle_detected = False
        for u, v, w in self.graph:
            if dist[u] != INF and dist[u] + w < dist[v]:
                negative_cycle_detected = True
                break

        if negative_cycle_detected:
            print("Graph contains a negative weight cycle!")
            return None, None # Return None to indicate cycle
        else:
            print("Graph does not contain a negative weight cycle.")
            return dist, pred


In [ ]:

graph1 = Graph(['A', 'B', 'C', 'D', 'E'])
graph1.add_edge('A', 'B', 1)
graph1.add_edge('A', 'C', 4)
graph1.add_edge('B', 'C', 2)
graph1.add_edge('B', 'D', 5)
graph1.add_edge('C', 'D', 1)
graph1.add_edge('D', 'E', 3)

source_node = 'A'
distances1, predecessors1 = graph1.bellman_ford(source_node)

if distances1:
    print(f"\nShortest distances from source '{source_node}':")
    for node, d in sorted(distances1.items()):
        print(f"  {source_node} -> {node}: {d}")
    print(f"\nPredecessors from source '{source_node}':")
    for node, p in sorted(predecessors1.items()):
        print(f"  {node}: {p}")

Graph does not contain a negative weight cycle.

Shortest distances from source 'A':
  A -> A: 0
  A -> B: 1
  A -> C: 3
  A -> D: 4
  A -> E: 7

Predecessors from source 'A':
  A: None
  B: A
  C: B
  D: C
  E: D
